# Day 26: Build a "Hierarchical Chunking" strategy

## Core Theory (Just-in-Time)

When building Retrieval-Augmented Generation (RAG) systems, you often face a fundamental trade-off when deciding on chunk size:
* **Small Chunks (e.g., 200 tokens):** Yield highly specific and accurate vector similarity matches, but they lack the surrounding context. When passed to the LLM, the LLM might struggle to synthesize a comprehensive answer.
* **Large Chunks (e.g., 2000 tokens):** Provide excellent context for the LLM, but they contain too much noise. The vector embeddings become diluted, leading to less precise retrieval.

**Hierarchical Chunking (also known as Parent-Child Chunking)** resolves this tension by giving you the best of both worlds:
1. **Parent Chunks:** Split documents into large "parent" chunks.
2. **Child Chunks:** Split those parent chunks into smaller "child" chunks.
3. **Storage:** Embed and store the *child* chunks in the vector database. Crucially, each child chunk maintains a reference (ID) to its parent chunk. The parent chunks are stored in a standard key-value document store.
4. **Retrieval:** During a query, you search against the *child* chunks for precise matching. Once the most relevant child chunks are identified, the system looks up their corresponding *parent* chunks and passes those larger context blocks to the LLM.

In the LangChain ecosystem, this pattern is elegantly handled by the `ParentDocumentRetriever`. It orchestrates the routing between a VectorStore (for children) and a BaseStore (for parents).



## Code Implementation


In [1]:
import os
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_core.embeddings import Embeddings
from langchain_openai import OpenAIEmbeddings

# For offline lab environment compatibility, check for an API key. 
# In a real environment, you must provide your actual OPENAI_API_KEY.
api_key_set = "OPENAI_API_KEY" in os.environ
if not api_key_set:
    print("Warning: OPENAI_API_KEY not found in environment. This script is intended to be run with a valid API key.")

def build_hierarchical_retriever(
    documents: List[Document],
    collection_name: str = "hierarchical_chunks",
    embeddings: Embeddings = None
) -> ParentDocumentRetriever:
    """
    Builds a ParentDocumentRetriever using Qdrant for child chunks and an InMemoryStore for parent chunks.
    
    Args:
        documents: A list of LangChain Document objects to index.
        collection_name: The name of the Qdrant collection.
        embeddings: Production-grade Embeddings instance. Defaults to OpenAIEmbeddings.
        
    Returns:
        An initialized ParentDocumentRetriever.
    """
    if embeddings is None:
        if api_key_set:
            embeddings = OpenAIEmbeddings()
        else:
            # We fail-fast if no key is provided to avoid dummy mock objects per user specifications
            raise ValueError("OPENAI_API_KEY not found, cannot initialize OpenAIEmbeddings")

    # 1. Initialize Qdrant Client for child chunks
    client = QdrantClient(location=":memory:")
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
    )
    
    # 2. Setup VectorStore and Embeddings
    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings
    )
    
    # 3. Setup Document Store for parent documents
    store = InMemoryStore()
    
    # 4. Define Chunkers
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
    
    # 5. Initialize the Retriever
    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=store,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )
    
    # 6. Add documents to the retriever
    retriever.add_documents(documents)
    
    return retriever

# Example Usage
if __name__ == "__main__":
    sample_text = (
        "AI Engineering is a rapidly growing field. It focuses on taking AI models "
        "and putting them into production safely. " * 20
        + "Another important aspect is data quality. Without good data, models "
        "will hallucinate or perform poorly. " * 20
    )
    
    docs = [Document(page_content=sample_text, metadata={"source": "ai_handbook.txt"})]
    
    if api_key_set:
        print("Building hierarchical retriever and indexing documents...")
        try:
            retriever = build_hierarchical_retriever(docs)
            query = "What causes models to hallucinate?"
            print(f"Querying for: '{query}'")
            results = retriever.invoke(query)
            
            print(f"Retrieved {len(results)} parent chunks.")
            if results:
                print(f"Content length of first result: {len(results[0].page_content)}")
                print(f"Snippet of first result: {results[0].page_content[:100]}...")
        except Exception as e:
             print(f"Error during retrieval example: {e}")
    else:
        print("Skipping example execution. Set OPENAI_API_KEY to run.")




Skipping example execution. Set OPENAI_API_KEY to run.


## Practical Lab / Homework

**Your Task:**
1. Read a real document (like a small PDF or text file) using a LangChain document loader.
2. Implement the `ParentDocumentRetriever` using Qdrant as the vector store and `InMemoryStore` as the docstore.
3. Query the retriever with a specific question and verify that the returned chunk is the *parent* chunk (e.g., check the string length or print the output).
4. **Bonus:** Try modifying the `parent_splitter` to return full documents instead of chunks (by omitting the `parent_splitter` argument or setting its chunk size to be very large), and observe how the results change.

Below is the implementation skeleton of the lab task. Review the implementation and execute it if you provide an API key.



In [2]:
# LAB TASK SOLUTION
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os

api_key_set = "OPENAI_API_KEY" in os.environ

if api_key_set:
    # 1. Create a document directly for the lab task
    lab_docs = [
        Document(
            page_content="""
            Chapter 1: The Beginning. The quick brown fox jumps over the lazy dog.
            This is a very important sentence about the fox.
            
            Chapter 2: The Middle. The dog was not amused. It barked loudly.
            
            Chapter 3: The End. They all went to sleep.
            """
        )
    ]
    
    # 2. Setup the retriever using Qdrant and InMemoryStore
    lab_client = QdrantClient(location=":memory:")
    lab_client.create_collection(
        collection_name="lab_collection",
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
    )
    
    lab_vectorstore = QdrantVectorStore(
        client=lab_client,
        collection_name="lab_collection",
        embedding=OpenAIEmbeddings()
    )
    
    lab_store = InMemoryStore()
    
    lab_parent_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)
    lab_child_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
    
    lab_retriever = ParentDocumentRetriever(
        vectorstore=lab_vectorstore,
        docstore=lab_store,
        child_splitter=lab_child_splitter,
        parent_splitter=lab_parent_splitter,
    )
    
    lab_retriever.add_documents(lab_docs)
    
    # 3. Query the retriever
    query = "What did the dog do?"
    results = lab_retriever.invoke(query)
    
    # 4. Print results to verify you got the parent context
    print("LAB RESULTS:")
    for idx, res in enumerate(results):
        print(f"Result {idx + 1} (Length {len(res.page_content)}):\n{res.page_content.strip()}\n")
else:
    print("LAB RESULTS: Cannot execute query without OPENAI_API_KEY")



LAB RESULTS: Cannot execute query without OPENAI_API_KEY


## Common Pitfalls

1. **Storage De-synchronization:** When deleting documents, you must remember to delete *both* the parent document from the DocStore and the associated child embeddings from the VectorStore. If they get out of sync, you may retrieve child IDs that don't map to any parent.
2. **Overlap Tuning:** While child chunks should have overlap to maintain context across chunk boundaries, parent chunks often need less or no overlap, as they are already large. Tuning these overlap parameters incorrectly can lead to duplicating large amounts of text in your DocStore.
3. **Retrieval Latency:** Hierarchical chunking requires a two-step retrieval process (Vector search -> DocStore lookup). If your DocStore is slow (e.g., a poorly indexed relational database or high-latency network store), this will significantly slow down your RAG pipeline. Redis or fast key-value stores are recommended for production DocStores.
4. **Memory Exhaustion:** Using `InMemoryStore` is great for local testing and notebooks, but it is not ephemeral between restarts. Always use a persistent store (like RedisStore or a persistent database) in production to avoid losing your parent mappings.

